# Module 15 — Type Hints and Data Modelling

**Part IV: Advanced** · Level: Advanced · Prerequisites: Modules 8, 14

## What you'll learn

- Annotating variables, parameters, and return values
- The `typing` module: `Optional`, `Union`, `Callable`, and generics
- Data containers with `dataclasses`
- `namedtuple` and `typing.NamedTuple`

---

## 15.1 Why type hints?

Python is dynamically typed, and hints are **not enforced at runtime**. They document intent, enable editor autocomplete, and let static checkers such as `mypy` or Pyright find bugs before you run the code. In larger code bases they pay for themselves quickly.

In [ ]:
def add(a: int, b: int) -> int:
    return a + b

print(add(2, 3))
print(add("x", "y"))      # runs fine! the hint is not enforced -> 'xy'

## 15.2 Annotating variables and functions

In [ ]:
retries: int = 3
ratio: float = 0.75
name: str = "report"
enabled: bool = True
tags: list[str] = ["a", "b"]                   # built-in generics: Python 3.9+
point: tuple[int, int] = (0, 1)                # fixed-length tuple
scores: dict[str, int] = {"ada": 90}
unique: set[int] = {1, 2, 3}
anything: tuple[int, ...] = (1, 2, 3)          # variable-length tuple

def append_greeting(names: list[str], greeting: str = "Hello") -> list[str]:
    """Return a greeting for every name."""
    return [f"{greeting}, {n}" for n in names]

print(append_greeting(["Ada", "Alan"]))
print(append_greeting.__annotations__)

## 15.3 The `typing` module

In [ ]:
from typing import Optional, Union, Callable, Iterable, Any

def find_user(user_id: int) -> Optional[str]:         # str or None
    users = {1: "ada", 2: "grace"}
    return users.get(user_id)

def parse(value: Union[int, str]) -> int:               # int or str
    return int(value)

# Python 3.10+ shorthand for the two above:
def find_user_v2(user_id: int) -> str | None:
    return {1: "ada"}.get(user_id)

def apply_all(funcs: list[Callable[[int], int]], x: int) -> list[int]:
    return [f(x) for f in funcs]

def total(values: Iterable[float]) -> float:             # accepts any iterable
    return sum(values)

def summarize(counts: dict[str, int], label: Optional[str] = None) -> list[str]:
    return [f"{label}: {k}={v}" if label else f"{k}={v}" for k, v in counts.items()]

print(find_user(1), find_user(9), parse("42"), find_user_v2(1))
print(apply_all([lambda n: n + 1, lambda n: n * 2], 10))
print(total(x / 2 for x in range(4)))
print(summarize({"00": 512}, label="run"))

Other useful names: `Any`, `Literal["a", "b"]`, `Final`, `TypedDict`, `Protocol`, and `TypeVar` for generic functions.

In [ ]:
from typing import TypeVar, Literal

T = TypeVar("T")

def first(items: list[T]) -> T:           # return type matches element type
    return items[0]

Mode = Literal["fast", "safe"]

def run(mode: Mode) -> str:
    return f"running in {mode} mode"

print(first([10, 20]), first(["a", "b"]), run("fast"))

## 15.4 Dataclasses

A `@dataclass` generates `__init__`, `__repr__`, and `__eq__` for you from annotated fields. It is the quickest way to write a clean data container.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class RunResult:
    backend: str
    items: int
    tags: list[str] = field(default_factory=list)    # NEVER write tags: list = []

r = RunResult("local", 1024)
r.tags.append("test")
print(r)
print(r == RunResult("local", 1024, ["test"]))       # True: generated __eq__

Useful options:

In [ ]:
from dataclasses import dataclass, field, asdict, replace

@dataclass(frozen=True, order=True)        # immutable, hashable, sortable
class Version:
    major: int
    minor: int = 0

v1, v2 = Version(1, 2), Version(1, 10)
print(v1 < v2, v1)
# v1.major = 5    # FrozenInstanceError
print(replace(v1, minor=3))                # make a modified copy

@dataclass
class Order:
    id: int
    items: list[str] = field(default_factory=list)
    total: float = field(init=False, default=0.0)    # computed, not a constructor arg

    def __post_init__(self):                         # runs right after __init__
        self.total = 9.99 * len(self.items)

order = Order(1, ["pen", "ink"])
print(order, asdict(order))

Mutable defaults must use `field(default_factory=...)`; dataclasses raise an error if you try a bare list or dict.

### Dataclass vs regular class

Use a dataclass when the class is mostly *data*. Write a regular class when it is mostly *behaviour* or needs unusual construction. Dataclasses can still have methods, properties, and inheritance.

## 15.5 Named tuples

A **named tuple** is an immutable, lightweight record with named fields — still a tuple, so it unpacks and indexes.

In [ ]:
from collections import namedtuple
from typing import NamedTuple

Point = namedtuple("Point", ["x", "y"])
p = Point(1, 2)
print(p.x, p.y, p[0], p)
x, y = p                                     # unpacking works
print(p._asdict(), p._replace(x=10))

class Employee(NamedTuple):                  # typed version with a class body
    name: str
    salary: float = 0.0

e = Employee("Ada", 100.0)
print(e, e.name)

| Need | Choose |
|---|---|
| Mutable record, defaults, methods | `@dataclass` |
| Immutable and hashable | `@dataclass(frozen=True)` or `NamedTuple` |
| Return several values from a function | `NamedTuple` |
| Quick dictionary-shaped data | `dict` / `TypedDict` |

## Common pitfalls

- Believing hints enforce types at runtime.
- Using a mutable default in a dataclass without `default_factory`.
- Over-annotating trivial local variables; annotate **interfaces** (function signatures, public attributes) first.
- Using `typing.List`/`Dict` in new code — the lowercase built-ins (`list[str]`) are preferred in 3.9+.

## Practice

1. Annotate every function you wrote in Module 8 and describe what a checker could now catch.
2. Create a `Book` dataclass with title, author, and a list of tags; sort a list of books by title.
3. Replace a function that returns a bare tuple `(name, age, city)` with one that returns a `NamedTuple`.